In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root       5630 F.... wandb-xpu
/dev/nvidiactl:      root       5630 F.... wandb-xpu
/dev/nvidia-uvm:     root       5630 F.... wandb-xpu


In [2]:
# Check GPU status
!nvidia-smi

Sun Jul 12 10:55:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P8             21W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
import os
os.environ['UNSLOTH_DISABLE_COMPILATION'] = '1'

In [ ]:
%%capture
import re
if 'COLAB_' not in ''.join(os.environ.keys()):
    !uv pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !uv pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !uv pip install --no-deps --upgrade "torchao>=0.16.0"
# !uv pip install --upgrade transformers
!uv pip install --no-deps trl==0.22.2

In [5]:
import unsloth
import os
import math
import types
import json
import torch
from datetime import datetime
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, EarlyStoppingCallback
from peft import PeftModel
from datasets import load_dataset, Dataset
from trl import SFTConfig, SFTTrainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Utilities

In [6]:
def load_train_val_datasets(
    lang, # e.g., 'en' | 'ja' | 'id'
    task, # 'wikipedia' | 'squad'
    train_size, val_size,
):
    # Validate that if the task is 'squad', the language must be 'en'
    if task == 'squad':
        assert lang == 'en', "SQuAD is English-only."
    
    # Define dataset configurations for each task
    data_configs = {
        'wikipedia': {
            'data_id': 'wikimedia/wikipedia',
            'data_dir': f'20231101.{lang}',
            'train_split': 'train',
            'val_split': 'train',
        },
        'squad': {
            'data_id': 'rajpurkar/squad',
            'data_dir': None,
            'train_split': 'train',
            'val_split': 'validation',
        },
    }
    
    # Validate that the specified task is supported
    assert task in data_configs, (
        f"Unsupported task: {task}. "
        f"Supported tasks: {list(data_configs.keys())}"
    )

    # Set up Hugging Face dataset configuration
    data_id = data_configs[task]['data_id']
    data_dir = data_configs[task]['data_dir']
    train_split = data_configs[task]['train_split']
    val_split = data_configs[task]['val_split']

    if train_split == val_split:
        # If the train and validation splits are the same, we need to sample from the same dataset stream
        dataset_stream = load_dataset(
            data_id,
            data_dir=data_dir,
            split=train_split,
            streaming=True,
        )

        train_data = []
        val_data = []

        for i, example in enumerate(dataset_stream):
            if i < train_size:
                train_data.append(example)
            elif i < train_size + val_size:
                val_data.append(example)
            else:
                break

    else:
        # If the train and validation splits are different, we can sample from each split separately
        def sample_split(split, size):
            dataset_stream = load_dataset(
                data_id,
                data_dir=data_dir,
                split=split,
                streaming=True,
            )

            data = []
            for i, example in enumerate(dataset_stream):
                if i >= size:
                    break
                data.append(example)
            return data

        train_data = sample_split(train_split, train_size)
        val_data = sample_split(val_split, val_size)

    return (
        Dataset.from_list(train_data),
        Dataset.from_list(val_data),
    )

# Configurations

In [7]:
# Run configuration
SEED = 42
USERNAME = 'alxxtexxr'
LANG = 'en'  # e.g., 'en' | 'ja' | 'id'
TASK = 'squad'  # 'wikipedia' | 'squad'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
MODEL_ID = 'unsloth/Qwen3.5-0.8B'
MODEL_NAME = 'Qwen3.5-0.8B'
RESUME_MODEL_ID = None
RESUME_CKPT_STEP = None
MAX_SEQ_LENGTH = 2048

# LoRA configuration
LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0
LORA_TARGET_MODULES = 'all-linear'

# Data configuration
TRAIN_SIZE = 1000
VAL_SIZE = 125

# Training configuration
MINI_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 20
WARMUP_STEPS = 50
LR = 2e-4

In [8]:
# Resume training configuration
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    model_name = RESUME_MODEL_ID
    run_name = model_name.split('/')[-1]
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_CKPT_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist."
            
else:
    run_name = f'{MODEL_NAME}-{TASK}-{LANG}-{TRAIN_SIZE/1000:g}K-LoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    hub_model_id = f'{USERNAME}/{run_name}'
base_hub_model_id, version = hub_model_id.rsplit('-v', 1)
hub_merged_model_id = f'{base_hub_model_id}-Merged-v{version}'

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", MODEL_NAME)
print("Run name:", run_name)
print("Hub model ID:", hub_model_id)
print("Hub merged model ID:", hub_merged_model_id)

Resume from checkpoint: False
Model name: Qwen3.5-0.8B
Run name: Qwen3.5-0.8B-squad-en-1K-LoRA-v260712105551
Hub model ID: alxxtexxr/Qwen3.5-0.8B-squad-en-1K-LoRA-v260712105551
Hub merged model ID: alxxtexxr/Qwen3.5-0.8B-squad-en-1K-LoRA-Merged-v260712105551


In [9]:
# Set environment variables for wandb logging
os.environ['WANDB_PROJECT'] = 'legamex'
os.environ['WANDB_NAME'] = run_name
# os.environ['WANDB_LOG_MODEL'] = 'checkpoint' # Control whether checkpoints get uploaded to wandb as artifacts

# Model

In [10]:
# Load model with Unsloth, but discard the processor it returns
model, _ = FastLanguageModel.from_pretrained(
    MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False,
    load_in_8bit = True,
)

# Load a plain text tokenizer directly from the original model repo
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if resume_from_checkpoint:
    # Load the LoRA adapter from the checkpoint and ensure it's in training mode
    model = PeftModel.from_pretrained(model, resume_from_checkpoint)
    model.inference_mode = False  # Disable inference-only flag
    model.enable_adapter_layers() # Explicitly unfreeze LoRA weights
else:
    # Initialize the LoRA adapter
    model = FastLanguageModel.get_peft_model(
        model,
        random_state=SEED,
        target_modules=LORA_TARGET_MODULES,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        use_gradient_checkpointing='unsloth', # True or 'unsloth' for very long context
        use_rslora=False,
        loftq_config=None,
    )
model = model.to(DEVICE)

# Prepare the model for training and set the pad token
model = FastLanguageModel.for_training(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.print_trainable_parameters()
print("device:", model.device)

==((====))==  Unsloth 2026.7.2: Fast Qwen3_5 patching. Transformers: 5.12.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

trainable params: 13,181,952 || all params: 866,167,872 || trainable%: 1.5219
device: cuda:0


In [ ]:
# Make the config JSON‑safe for logging without affecting the model itself
original_to_json = model.config.to_json_string

def safe_to_json_string(self, use_diff=True):
    try:
        return original_to_json(use_diff=use_diff)
    except (TypeError, ValueError):
        # Return a minimal, valid JSON string if serialization fails
        return json.dumps({"info": "config contains non-serializable objects"})

model.config.to_json_string = types.MethodType(safe_to_json_string, model.config)

# Data

In [11]:
# Load the dataset
train_dataset, val_dataset = load_train_val_datasets(lang=LANG, task=TASK, 
                                                     train_size=TRAIN_SIZE, val_size=VAL_SIZE)

print("Train dataset:")
print(train_dataset)
print()
print("Validation dataset:")
print(val_dataset)

Train dataset:
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 1000
})

Validation dataset:
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 125
})


# Training

In [ ]:
# Calculate the maximum number of training steps
max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

# Define a formatting function for the trainer
def formatting_prompts_func(examples):
    # Convert a single example to a batch (dict of lists) for uniform handling
    if not isinstance(examples['context'], list):
        examples = {k: [v] for k, v in examples.items()}

    outputs = []
    for i in range(len(examples['context'])):
        answer_text = examples['answers'][i]['text'][0] # first answer
        messages = [
            {"role": "user", "content": f"Context: {examples['context'][i]}\n\nQuestion: {examples['question'][i]}"},
            {"role": "assistant", "content": answer_text}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        outputs.append(text)
    return outputs  # Always a list of strings

# Set up the trainer
training_args = SFTConfig(
    # Training arguments
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps = max_steps,
    warmup_steps = WARMUP_STEPS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=1.0,
    weight_decay=0.01,
    
    # Validation arguments
    eval_strategy='steps',
    eval_steps=20,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps=10,
    # logging_first_step=True,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    save_steps=20,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. WARN: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better = False,

    # run_name=run_name,
    output_dir=run_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,
    hub_strategy='all_checkpoints',
    hub_always_push=True,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=formatting_prompts_func,
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            # early_stopping_threshold = 0.001,
        )
    ],
)

Calculated max steps: 1260
Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/125 [00:00<?, ? examples/s]

In [16]:
# Start training
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 20 | Total steps = 1,260
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 13,181,952 of 866,167,872 (1.52% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
20,2.452569,1.941546
40,2.128618,1.973444
60,1.717780,2.116493
80,1.210903,2.495487


# Merging

In [17]:
# After training finishes, merge LoRA into the base model and save everything
model.eval() # Good practice
merged_model = model.merge_and_unload() # Merge LoRA + base, returns a plain model

# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:97: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


README.md:   0%|          | 0.00/535 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

README.md:   0%|          | 0.00/541 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpsski3yeq/tokenizer.json:  40%|####      | 8.00MB / 20.0MB            

Merged model uploaded to: https://huggingface.co/alxxtexxr/Qwen3.5-0.8B-squad-en-1K-LoRA-Merged-v260712105551
